In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

taxiDB = pd.read_csv('data/taxi_dataset.csv')

In [2]:
taxiDB.head(5)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N


<dl>
<dt> Описание колонок:
<dd>id - ID поездки </dd>
<dd>vendor_id - ID компании, осуществляющей перевозку </dd>
<dd>pickup_datetime - Таймкод начала поездки</dd>
<dd>dropoff_datetime - Таймкод конца поездки </dd>
<dd>passenger_count - Количество пассажиров </dd>
<dd>pickup_longitude - Долгота точки, в которой началась поездка </dd>
<dd>pickup_latitude - Широта точки, в которой началась поездка </dd>
<dd>dropoff_longitude - Долгота точки, в которой закончилась поездка </dd>
<dd>dropoff_latitude - Широта точки, в которой закончилась поездка </dd>
<dd>store_and_fwd_flag - Yes/No: Была ли информация сохранена в памяти транспортного средства из-за потери соединения с сервером </dd>
</dl>

**Наша целевая переменная - длительность поездки.**

Зная тайм-коды времени начала и конца поездок, можем вычислить обозначенный таргет. Производим вычисления в секундах.
Важная ссылка <a href="https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html">данный способ</a> для перевода строки в datetime тип, с которым удобно работать при вычленении дней/часов

И <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.total_seconds.html"> этот </a>для перевода разницы datetime объектов в секунды

Положим таргетную переменнул в столбик с названием **trip_duration**

In [3]:
### переводим в datetime колонки начала и завершения поездки
taxiDB['pickup_datetime']=pd.to_datetime(taxiDB['pickup_datetime'])
taxiDB['dropoff_datetime']=pd.to_datetime(taxiDB['dropoff_datetime'])
taxiDB.dtypes

id                            object
vendor_id                      int64
pickup_datetime       datetime64[ns]
dropoff_datetime      datetime64[ns]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag            object
dtype: object

In [4]:
### создаем таргетную переменную
taxiDB['trip_duration']=(taxiDB['dropoff_datetime']-taxiDB['pickup_datetime']).dt.seconds
taxiDB['trip_duration']

0           455
1           663
2          2124
3           429
4           435
           ... 
1458639     778
1458640     655
1458641     764
1458642     373
1458643     198
Name: trip_duration, Length: 1458644, dtype: int32

In [5]:
taxiDB.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


Предсказывая таргет для новых объектов в будущем, мы не будем заранее знать **dropoff_datetime**.

Удалим колонку из датасета.

In [6]:
### удалаяем колонку
taxiDB=taxiDB.drop("dropoff_datetime", axis=1)
taxiDB.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,1,-73.973053,40.793209,-73.972923,40.782520,N,435



**Рассмотрим имеющиеся вещественные/бинарные, какие простейшие признаки можно вытащить из остальных колонок.**

Во-первых, имеем бинарный признак vendor_id, принимающий значения {1, 2}. Переведем его во множество {0, 1}, так как это просто привычнее.

In [7]:
taxiDB['vendor_id'] = taxiDB['vendor_id'] - 1
taxiDB.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,1,2016-03-14 17:24:55,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,0,2016-06-12 00:43:35,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,1,2016-01-19 11:35:24,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,1,2016-04-06 19:32:31,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,1,2016-03-26 13:30:55,1,-73.973053,40.793209,-73.972923,40.782520,N,435


Подозреваем колонку store_and_fwd_flag в бинарности. Закодирую во множество {0, 1}.

In [8]:
### Your code is here
taxiDB['store_and_fwd_flag'].value_counts()

store_and_fwd_flag
N    1450599
Y       8045
Name: count, dtype: int64

In [9]:
### 1 вариант
taxiDB['store_and_fwd_flag'] = taxiDB['store_and_fwd_flag'].apply(lambda x: 0 if x == 'N' else 1)
### 2 вариант
### taxiDB.loc[taxiDB['store_and_fwd_flag'] == 'N', 'store_and_fwd_flag'] =0
### taxiDB.loc[taxiDB['store_and_fwd_flag'] != 'N', 'store_and_fwd_flag'] =1
taxiDB['store_and_fwd_flag']

0          0
1          0
2          0
3          0
4          0
          ..
1458639    0
1458640    0
1458641    0
1458642    0
1458643    0
Name: store_and_fwd_flag, Length: 1458644, dtype: int64

Во-вторых, можем использовать долготу и широту точек старта/завершения поездки, чтобы примерно оценить расстояние между 2 точками.

Сами по себе они, как самостоятельные вещественные признаки, вряд ли способны хорошо объяснять длительность поездки.

Базовая идея состоит в том, чтобы посчитать разность долгот и широт соответственно, то есть:

$$
\delta_{long} = \mathrm{dropoff\_longitude} - \mathrm{pickup\_longitude}
$$

$$
\delta_{lat} = \mathrm{dropoff\_latitude} - \mathrm{pickup\_latitude}
$$

А потом вычислить географическое расстояние между 2 точками по теореме Пифагора:

$$
R = \sqrt{\delta_{long}^2 + \delta_{lat}^2}
$$

Реализуем данную задумку и вычислим такую вещественную колонку **R**, что, в целом, является хорошим тоном при работе с координатами точек.

Только для начала нужно некоторым образом перевести долготу и широту в километры, обеспечив равенство их мер измерения. Потому что, вообще говоря, *градусная мера* широт и долгот имеет неодинаковую шкалу перевода в километры. Так, если пропустить данную деталь, расстояние **R** будет вычислено неверно, ведь катеты тогда будут иметь разную размерность.

В целом, перевод из долгот и широт в расстояние поездки позволяет нам в будущем проверить зависимость **длительности поездки от километража**, и объяснить ее будет куда проще, чем аналогичную между таргетом и изначальными признаками

<a href="https://www.datafix.com.au/BASHing/2018-11-07.html"> Маленькая статья про перевод разницы градусов долгот/широт в километры</a>

**Начнем переводить каждую долготу в некоторое относительно километровое выражение**

Соберем список из всех широт (как точек старта, так и конца).

In [10]:
allLat  = list(taxiDB['pickup_latitude']) + list(taxiDB['dropoff_latitude'])


Посчитаем медиану:

Это некоторое "Центральное значение" в отсортированном массиве всех значений.

Иными словами, такое число, меньше и больше которого примерно равное количество объектов.

In [11]:
medianLat  = sorted(allLat)[int(len(allLat)/2)]
medianLat

40.75431823730469

Теперь, для из каждого значения широты вычтем медианное значение.

Результат переведем в километры.

In [12]:
latMultiplier  = 111.32

taxiDB['pickup_latitude']   = latMultiplier  * (taxiDB['pickup_latitude']   - medianLat)
taxiDB['dropoff_latitude']   = latMultiplier  * (taxiDB['dropoff_latitude']  - medianLat)

In [13]:
taxiDB.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,1,2016-03-14 17:24:55,1,-73.982155,1.516008,-73.964630,1.256121,0,455
1,id2377394,0,2016-06-12 00:43:35,1,-73.980415,-1.753813,-73.999481,-2.578912,0,663
2,id3858529,1,2016-01-19 11:35:24,1,-73.979027,1.070973,-74.005333,-4.923841,0,2124
3,id3504673,1,2016-04-06 19:32:31,1,-74.010040,-3.823568,-74.012268,-5.298809,0,429
4,id2181028,1,2016-03-26 13:30:55,1,-73.973053,4.329328,-73.972923,3.139453,0,435


Итого, для **latitude** колонок получили следующие выражения:

*На сколько примерно километров севернее или южнее (в зависимости от знака) точка находится относительно средней широты*

In [14]:
allLong = list(taxiDB['pickup_longitude']) + list(taxiDB['dropoff_longitude'])

medianLong  = sorted(allLong)[int(len(allLong)/2)]

longMultiplier = np.cos(medianLat*(np.pi/180.0)) * 111.32

In [15]:
medianLong

-73.98085021972656

Используя полученную медиану и множитель, на который стоит корректировать все долготы, получите корректные **longitude** признаки по аналогии.

In [16]:
### Your code is here
longMultiplier  = 111.32

taxiDB['pickup_longitude']   = longMultiplier  * (taxiDB['pickup_longitude']   - medianLong)
taxiDB['dropoff_longitude']   = longMultiplier  * (taxiDB['dropoff_longitude']  - medianLong)
taxiDB.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,1,2016-03-14 17:24:55,1,-0.145231,1.516008,1.805621,1.256121,0,455
1,id2377394,0,2016-06-12 00:43:35,1,0.048410,-1.753813,-2.074001,-2.578912,0,663
2,id3858529,1,2016-01-19 11:35:24,1,0.202984,1.070973,-2.725417,-4.923841,0,2124
3,id3504673,1,2016-04-06 19:32:31,1,-3.249438,-3.823568,-3.497435,-5.298809,0,429
4,id2181028,1,2016-03-26 13:30:55,1,0.867989,4.329328,0.882427,3.139453,0,435


Почему мы вычисляли через медианы: они позволяют нам во время вычисления расстояния преобразовать изначальные longtitude/latitude колонки в "отдаленности точек старта/конца поездок" от медианных точек. Кажется, что это прикольно :) Есть подозрение, что медианная для поездок точка города - это, на практике, точка скопления вечерних пробок. Нам может быть вполне важно знать, насколько далеко от такого эпицентра ужаса мы начинаем и заканчиваем поездку (насколько севернее/южнее/...) и выделить поверх этой информации дополнительные признаки.<br>
Это ещё один пример, как можно работать с признаками.

Наконец, вычислим географическое расстояние **distance_km**:

In [17]:
### Your code is here
delta_long = taxiDB['dropoff_longitude'] - taxiDB['pickup_longitude']

delta_lat =  taxiDB['dropoff_latitude'] - taxiDB['pickup_latitude']
taxiDB['distance_km'] = (delta_long**2+delta_lat**2)**0.5
#taxiDB['distance_km'] = ((taxiDB['dropoff_longitude'] - taxiDB['pickup_longitude'])**2 + 
                        #(taxiDB['dropoff_latitude'] - taxiDB['pickup_latitude'])**2)**0.5

In [18]:
taxiDB.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,distance_km
0,id2875421,1,2016-03-14 17:24:55,1,-0.145231,1.516008,1.805621,1.256121,0,455,1.968086
1,id2377394,0,2016-06-12 00:43:35,1,0.048410,-1.753813,-2.074001,-2.578912,0,663,2.277151
2,id3858529,1,2016-01-19 11:35:24,1,0.202984,1.070973,-2.725417,-4.923841,0,2124,6.671831
3,id3504673,1,2016-04-06 19:32:31,1,-3.249438,-3.823568,-3.497435,-5.298809,0,429,1.495941
4,id2181028,1,2016-03-26 13:30:55,1,0.867989,4.329328,0.882427,3.139453,0,435,1.189963


Уберем старые признаки!

In [19]:
taxiDB = taxiDB.drop(['pickup_longitude', 'dropoff_longitude',
                      'pickup_latitude', 'dropoff_latitude'], axis=1)

In [20]:
taxiDB.head()

,id,vendor_id,pickup_datetime,passenger_count,store_and_fwd_flag,trip_duration,distance_km
0,id2875421,1,2016-03-14 17:24:55,1,0,455,1.968086
1,id2377394,0,2016-06-12 00:43:35,1,0,663,2.277151
2,id3858529,1,2016-01-19 11:35:24,1,0,2124,6.671831
3,id3504673,1,2016-04-06 19:32:31,1,0,429,1.495941
4,id2181028,1,2016-03-26 13:30:55,1,0,435,1.189963


В-третьих, обратим внимание на колонку **passenger_count**.

In [21]:
### Your code is here
taxiDB['passenger_count'].value_counts()

passenger_count
1    1033540
2     210318
5      78088
3      59896
6      48333
4      28404
0         60
7          3
9          1
8          1
Name: count, dtype: int64

С одной стороны, можно воспринимать его как обычный вещественный признак. Ведь само по себе количество пассажиров (без дополнительной обработки) - это некоторое число, которое может принимать большое количество различных значений.

С другой стороны, мы знаем, что количество пассажиров от поездки к поездке ограничено. Вряд ли если к нам придут новые данные, мы увидим числа бОльшие, чем у нас в датасете. Тогда рассуждаем следующим образом: раз множество значений признака ограничено, то он категориальный (или, в данном случае, даже порядковый! Ведь у нас могут быть какие-то логичные предположения о том, что количество пассажиров может влиять на модель машины и, соответственно, скорость ее передвижения и скорость поездки!)

Какой подход выбрать лучше заранее наверняка не узнаешь. Нужны эксперименты с данными и моделями. Тем не менее, я предполагаю, что данный признак является категориальным!

Реализиуем прием с **Mean-target encoding'ом**. Заменим колонку **passenger_count** колонкой **category_encoded**.

In [22]:
### делаем MTE для колонки количества пассажирова в поездке, 
### чтобы посчитать среднее количество времени проведения в пути для каждой из групп пассажиров, то есть для 1, 2, 3 и т.д.
taxiDB['passenger_count'] = taxiDB['passenger_count'].map(taxiDB.groupby(['passenger_count'])['trip_duration'].mean())

In [23]:
### переименуем колонку passenger_count на category_encoded
taxiDB = taxiDB.rename(columns={'passenger_count': 'category_encoded'})

In [24]:
taxiDB.head(10)

,id,vendor_id,pickup_datetime,category_encoded,store_and_fwd_flag,trip_duration,distance_km
0,id2875421,1,2016-03-14 17:24:55,923.126885,0,455,1.968086
1,id2377394,0,2016-06-12 00:43:35,923.126885,0,663,2.277151
2,id3858529,1,2016-01-19 11:35:24,923.126885,0,2124,6.671831
3,id3504673,1,2016-04-06 19:32:31,923.126885,0,429,1.495941
4,id2181028,1,2016-03-26 13:30:55,923.126885,0,435,1.189963
5,id0801584,1,2016-01-30 22:01:40,1061.355223,0,443,1.288240
6,id1813257,0,2016-06-17 22:34:59,1053.529749,0,341,1.573305
7,id1324603,1,2016-05-21 07:54:58,923.126885,0,1551,6.657049
8,id1301050,0,2016-05-27 23:12:23,923.126885,0,255,1.646391
9,id0012891,1,2016-03-10 21:45:01,923.126885,0,1225,5.160199


In [25]:
###  Cохраните первые 10 значений полученного промежуточного датафрейма в файл в формате csv с сепаратором ';' Отправьте полученный файл в форму ответа задания 8.

Кажется, мы достаточно близки к тому, чтобы получить в итоге табличку, полностью состояющую из чиселок и, казалось бы, осмысленных признаков!

Остались две колонки: **id**, **pickup_datetime**

**id** можно использовать как обычный идентификатор нашего объекта, поэтому поместите данную колонку в качестве индекса нашей таблички:

In [26]:
taxiDB = taxiDB.set_index('id')


In [27]:
taxiDB.index = taxiDB.index.str.replace("id" , "").astype(int)

In [28]:
taxiDB.head()

,vendor_id,pickup_datetime,category_encoded,store_and_fwd_flag,trip_duration,distance_km
id,,,,,,
2875421,1,2016-03-14 17:24:55,923.126885,0,455,1.968086
2377394,0,2016-06-12 00:43:35,923.126885,0,663,2.277151
3858529,1,2016-01-19 11:35:24,923.126885,0,2124,6.671831
3504673,1,2016-04-06 19:32:31,923.126885,0,429,1.495941
2181028,1,2016-03-26 13:30:55,923.126885,0,435,1.189963
